# 02 — Exploratory Data Analysis

**Goal:** Understand the distributional characteristics and temporal behaviour of all six tickers before modelling.

Key questions we answer here:
- How did each ticker *relatively* perform from a common Jan 2018 baseline?
- How fat-tailed are the return distributions? (kurtosis / skewness)
- When did volatility spike — and which tickers were most affected?
- How correlated are returns across sectors?

**DuckDB SQL** drives all aggregations and ranking queries. Python handles what SQL cannot — rolling calculations and visualisation.

In [1]:
# ── 1. Imports ─────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import duckdb
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

# ── Shared style config ────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi'      : 150,
    'font.family'     : 'DejaVu Sans',
    'axes.spines.top' : False,
    'axes.spines.right': False,
    'axes.grid'       : True,
    'grid.alpha'      : 0.25,
    'grid.linestyle'  : ':',
})

DATA_DIR   = Path('data')
OUT_DIR    = Path('outputs')
OUT_DIR.mkdir(exist_ok=True)

COVID_DATE = pd.Timestamp('2020-03-20')
COLORS = {
    'AAPL': '#1565C0', 'AMZN': '#E65100', 'GLD' : '#2E7D32',
    'JPM' : '#C62828', 'SPY' : '#6A1B9A', 'XOM' : '#4E342E'
}
print('Imports OK')

Imports OK


In [2]:
# ── 2. Load data ───────────────────────────────────────────────────────────
df = pd.read_csv(DATA_DIR / 'market_data.csv', parse_dates=['Date'])
df = df.sort_values(['Ticker', 'Date']).reset_index(drop=True)
print(f'Loaded: {df.shape}  |  {df.Date.min().date()} → {df.Date.max().date()}')

Loaded: (12426, 10)  |  2018-01-02 → 2026-03-30


## SQL Block 1 — Annual Returns Per Ticker

Before plotting, we use DuckDB to get the **year-over-year return** for every ticker. This gives immediate context: which years were good, which were bad, and for whom?

In [3]:
# ── 3. DuckDB — annual returns per ticker ────────────────────────────────
con = duckdb.connect()
con.register('market', df)

annual = con.execute("""
    WITH yearly AS (
        SELECT
            Ticker,
            YEAR(CAST(Date AS DATE))                     AS yr,
            FIRST(Close ORDER BY Date ASC)               AS open_price,
            LAST(Close  ORDER BY Date ASC)               AS close_price
        FROM market
        GROUP BY Ticker, yr
    )
    SELECT
        Ticker,
        yr,
        ROUND(open_price,  2)                            AS open_price,
        ROUND(close_price, 2)                            AS close_price,
        ROUND((close_price - open_price) / open_price * 100, 2) AS annual_ret_pct
    FROM yearly
    ORDER BY Ticker, yr
""").df()

print('=== Annual Return % per Ticker (DuckDB) ===')
annual_pivot = annual.pivot(index='yr', columns='Ticker', values='annual_ret_pct')
print(annual_pivot.to_string())

=== Annual Return % per Ticker (DuckDB) ===
Ticker   AAPL   AMZN    GLD    JPM    SPY    XOM
yr                                              
2018    -7.05  26.32  -3.12  -7.50  -5.25 -16.48
2019    88.74  20.06  17.78  44.75  31.09   4.92
2020    78.24  71.60  23.90  -6.67  17.24 -37.22
2021    38.06   4.64  -6.24  28.96  30.51  56.51
2022   -28.20 -50.71   0.78 -14.45 -18.65  80.48
2023    54.80  77.04  11.76  29.64  26.71  -2.93
2024    35.56  46.33  26.96  42.63  25.59   8.67
2025    11.99   4.81  61.48  37.10  18.01  16.26
2026    -8.91 -11.28   4.09 -12.42  -7.24  40.74


## Chart 1 — Normalised Price (Log Scale)

Rebasing to 100 at Jan 2018 shows *percentage gains* from a common start. We use a **log y-axis** so that equal vertical distances represent equal *percentage moves* — otherwise AAPL and AMZN's extreme growth visually flattens every other ticker.

The vertical dashed red line marks the COVID crash bottom (2020-03-20).

In [4]:
# ── 4. Rebase to 100 and plot (log scale) ────────────────────────────────
first_close = (
    df.sort_values('Date').groupby('Ticker')['Close'].first().rename('FirstClose')
)
df = df.merge(first_close, on='Ticker')
df['Rebased'] = df['Close'] / df['FirstClose'] * 100
df = df.drop(columns='FirstClose')

fig, ax = plt.subplots(figsize=(14, 7))

for ticker in sorted(df['Ticker'].unique()):
    sub = df[df['Ticker'] == ticker].sort_values('Date')
    ax.plot(sub['Date'], sub['Rebased'],
            color=COLORS[ticker], linewidth=2.0, label=ticker, alpha=0.9)
    # End-of-line label
    last = sub.iloc[-1]
    ax.annotate(
        f"{ticker}  {last['Rebased']:.0f}",
        xy=(last['Date'], last['Rebased']),
        xytext=(6, 0), textcoords='offset points',
        fontsize=8.5, fontweight='bold',
        color=COLORS[ticker], va='center'
    )

# COVID line
ax.axvline(COVID_DATE, color='red', linestyle='--', linewidth=1.4, alpha=0.8)
ax.text(COVID_DATE - pd.Timedelta(days=30), ax.get_ylim()[1] * 0.96,
        'COVID\ncrash', color='red', fontsize=8.5, ha='right', va='top',
        bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))

# Log scale with clean tick labels
ax.set_yscale('log')
ticks = [75, 100, 150, 200, 300, 400, 600, 800]
ax.set_yticks(ticks)
ax.get_yaxis().set_major_formatter(mticker.ScalarFormatter())

ax.set_xlim(df['Date'].min() - pd.Timedelta(days=30),
            df['Date'].max() + pd.Timedelta(days=180))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.set_title('Normalised Price — Rebased to 100 at Jan 2018  (log scale)',
             fontsize=14, fontweight='bold', pad=14)
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Indexed Price — log scale  (100 = Jan 2018)', fontsize=11)
ax.legend(loc='upper left', fontsize=9, ncol=2, framealpha=0.9)
plt.tight_layout()
plt.savefig(OUT_DIR / 'normalised_price.png', dpi=150, bbox_inches='tight')
print('Saved → outputs/normalised_price.png')
plt.show()

Saved → outputs/normalised_price.png


## SQL Block 2 — Distribution Statistics

DuckDB computes per-ticker return statistics. We then supplement with exact kurtosis and skewness from pandas.

- **Skewness > 0** → longer right tail (more extreme up-days than down-days)
- **Excess kurtosis > 0** → fatter tails than a normal distribution — standard for equity returns

In [5]:
# ── 5. DuckDB — return distribution stats ────────────────────────────────
dist_stats = con.execute("""
    SELECT
        Ticker,
        COUNT(*)                                           AS trading_days,
        ROUND(AVG(DailyReturn)*252*100,   2)               AS ann_return_pct,
        ROUND(STDDEV(DailyReturn)*SQRT(252)*100, 2)        AS ann_vol_pct,
        ROUND(MIN(DailyReturn)*100, 2)                     AS worst_day_pct,
        ROUND(MAX(DailyReturn)*100, 2)                     AS best_day_pct,
        SUM(CASE WHEN DailyReturn < -0.02 THEN 1 ELSE 0 END) AS days_down_2pct,
        SUM(CASE WHEN DailyReturn >  0.02 THEN 1 ELSE 0 END) AS days_up_2pct
    FROM market
    WHERE DailyReturn IS NOT NULL
    GROUP BY Ticker
    ORDER BY ann_return_pct DESC
""").df()

print('=== Return Distribution Stats ===')
print(dist_stats.to_string(index=False))

# Pandas exact kurtosis & skewness
pivot = df.pivot(index='Date', columns='Ticker', values='DailyReturn').dropna()
stats_df = pd.DataFrame({
    'Excess Kurtosis': pivot.kurtosis(),
    'Skewness'       : pivot.skew()
}).round(3)
print('\n=== Kurtosis & Skewness (pandas) ===')
print(stats_df.to_string())

=== Return Distribution Stats ===
Ticker  trading_days  ann_return_pct  ann_vol_pct  worst_day_pct  best_day_pct  days_down_2pct  days_up_2pct
  AAPL          2070           26.74        30.63         -12.86         15.33           210.0         232.0
  AMZN          2070           20.71        34.32         -14.05         13.54           259.0         284.0
   JPM          2070           18.67        28.94         -14.96         18.01           165.0         175.0
   XOM          2070           17.61        30.10         -12.22         12.69           211.0         240.0
   GLD          2070           15.95        16.49         -10.27          6.36            50.0          56.0
   SPY          2070           13.83        19.31         -10.94         10.50            86.0          60.0

=== Kurtosis & Skewness (pandas) ===
        Excess Kurtosis  Skewness
Ticker                           
AAPL              6.430     0.156
AMZN              4.185     0.093
GLD               7.957    -0

## SQL Block 3 — Top 10 Best & Worst Days

A key question for any anomaly project: *which days were genuinely extreme?*

In [6]:
# ── 6. DuckDB — best and worst single days ───────────────────────────────
best_worst = con.execute("""
    WITH ranked AS (
        SELECT
            Date, Ticker,
            ROUND(DailyReturn * 100, 2) AS ret_pct,
            ROUND("^VIX", 1)            AS VIX,
            ROW_NUMBER() OVER (ORDER BY DailyReturn ASC)  AS worst_rank,
            ROW_NUMBER() OVER (ORDER BY DailyReturn DESC) AS best_rank
        FROM market
        WHERE DailyReturn IS NOT NULL
    )
    SELECT Date, Ticker, ret_pct, VIX,
           'WORST' AS direction, worst_rank AS rank
    FROM ranked WHERE worst_rank <= 10
    UNION ALL
    SELECT Date, Ticker, ret_pct, VIX,
           'BEST'  AS direction, best_rank  AS rank
    FROM ranked WHERE best_rank  <= 10
    ORDER BY direction, rank
""").df()

print('=== Top 10 Best & Worst Days (any ticker) ===')
for direction in ['WORST', 'BEST']:
    print(f'\n--- {direction} ---')
    print(best_worst[best_worst['direction']==direction]
          .drop(columns='direction').to_string(index=False))

=== Top 10 Best & Worst Days (any ticker) ===

--- WORST ---
      Date Ticker  ret_pct  VIX  rank
2020-03-16    JPM   -14.96 82.7     1
2022-04-29   AMZN   -14.05 33.4     2
2020-03-09    JPM   -13.55 54.5     3
2020-03-16   AAPL   -12.86 82.7     4
2020-03-09    XOM   -12.22 54.5     5
2020-03-12    XOM   -11.43 75.5     6
2020-03-16    SPY   -10.94 82.7     7
2020-03-18    JPM   -10.53 76.4     8
2026-01-30    GLD   -10.27 17.4     9
2020-03-18    XOM   -10.02 76.4    10

--- BEST ---
      Date Ticker  ret_pct  VIX  rank
2020-03-13    JPM    18.01 57.8     1
2025-04-09   AAPL    15.33 33.6     2
2020-11-09    JPM    13.54 25.8     3
2022-02-04   AMZN    13.54 23.2     4
2020-03-24    XOM    12.69 61.7     5
2020-11-09    XOM    12.66 25.8     6
2022-11-10   AMZN    12.18 23.5     7
2020-03-13   AAPL    11.98 57.8     8
2025-04-09   AMZN    11.98 33.6     9
2020-03-24    JPM    11.89 61.7    10


## Chart 2 — Rolling 30-Day Annualised Volatility (Per-Ticker Subplots)

Putting all 6 tickers on one chart creates an unreadable tangle — especially with the COVID spike dominating the y-axis. **Separate subplots** give each ticker its own scale and make regime changes clearly visible.

In [7]:
# ── 7. Rolling volatility — 2×3 subplot grid ─────────────────────────────
df_v = df.sort_values(['Ticker', 'Date']).copy()
df_v['RolVol30'] = (
    df_v.groupby('Ticker')['DailyReturn']
        .transform(lambda x: x.rolling(30).std() * np.sqrt(252) * 100)
)

tickers = sorted(df_v['Ticker'].unique())
fig, axes = plt.subplots(2, 3, figsize=(16, 8), sharey=False)
axes = axes.flatten()

for i, ticker in enumerate(tickers):
    ax = axes[i]
    sub = df_v[df_v['Ticker'] == ticker].dropna(subset=['RolVol30'])

    # Fill + line
    ax.fill_between(sub['Date'], sub['RolVol30'],
                    alpha=0.20, color=COLORS[ticker])
    ax.plot(sub['Date'], sub['RolVol30'],
            color=COLORS[ticker], linewidth=1.5)

    # COVID reference line
    ax.axvline(COVID_DATE, color='red', linestyle='--', linewidth=1.1, alpha=0.8)
    ymax = sub['RolVol30'].quantile(0.99)          # clip extreme outliers for scale
    ax.set_ylim(0, ymax * 1.15)
    ax.set_title(ticker, fontweight='bold', fontsize=12, color=COLORS[ticker])
    ax.set_ylabel('Vol (%) annualised', fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator())
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right', fontsize=8)

    # Annotate peak
    peak_idx = sub['RolVol30'].idxmax()
    peak_row = sub.loc[peak_idx]
    ax.annotate(
        f"Peak\n{peak_row['RolVol30']:.0f}%",
        xy=(peak_row['Date'], peak_row['RolVol30']),
        xytext=(0, 8), textcoords='offset points',
        fontsize=7, ha='center', color='red',
        arrowprops=dict(arrowstyle='->', color='red', lw=0.8)
    )

fig.suptitle('Rolling 30-Day Annualised Volatility per Ticker\n'
             '(Red dashed = COVID crash  |  Scale independent per panel)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUT_DIR / 'rolling_volatility.png', dpi=150, bbox_inches='tight')
print('Saved → outputs/rolling_volatility.png')
plt.show()

Saved → outputs/rolling_volatility.png


## Chart 3 — Return Correlation Heatmap

Pairwise daily-return correlation across all tickers. GLD (gold) typically shows low or negative correlation with equities — the classic portfolio hedge. SPY and JPM high correlation is expected (banks are S&P constituents).

In [8]:
# ── 8. Seaborn correlation heatmap ────────────────────────────────────────
corr = pivot.corr().round(2)

fig3, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='RdYlGn',
    vmin=-1, vmax=1, linewidths=0.6,
    square=True, ax=ax,
    annot_kws={'size': 10, 'weight': 'bold'}
)
ax.set_title('Daily Return Correlation\n(2018 → present)', fontsize=13,
             fontweight='bold', pad=12)
ax.set_xlabel('Ticker', fontsize=10)
ax.set_ylabel('Ticker', fontsize=10)
plt.tight_layout()
plt.savefig(OUT_DIR / 'correlation_heatmap.png', dpi=150, bbox_inches='tight')
print('Saved → outputs/correlation_heatmap.png')
plt.show()

Saved → outputs/correlation_heatmap.png


## SQL Block 4 — Monthly Return Heatmap Data

A classic analyst view: how did each ticker perform in each calendar month? SQL `PIVOT`-style aggregation followed by a seaborn heatmap.

In [9]:
# ── 9. DuckDB — monthly return table ─────────────────────────────────────
monthly = con.execute("""
    SELECT
        Ticker,
        YEAR(CAST(Date AS DATE))                          AS yr,
        MONTH(CAST(Date AS DATE))                         AS mo,
        ROUND(SUM(DailyReturn) * 100, 2)                  AS total_ret_pct
    FROM market
    WHERE DailyReturn IS NOT NULL
    GROUP BY Ticker, yr, mo
    ORDER BY Ticker, yr, mo
""").df()

# Plot SPY monthly heatmap as an example
spy_monthly = monthly[monthly['Ticker'] == 'SPY'].copy()
spy_pivot   = spy_monthly.pivot(index='yr', columns='mo', values='total_ret_pct')
spy_pivot.columns = ['Jan','Feb','Mar','Apr','May','Jun',
                     'Jul','Aug','Sep','Oct','Nov','Dec']

fig4, ax4 = plt.subplots(figsize=(14, 5))
sns.heatmap(
    spy_pivot, annot=True, fmt='.1f', cmap='RdYlGn',
    center=0, linewidths=0.4, ax=ax4,
    annot_kws={'size': 8},
    cbar_kws={'label': 'Monthly Return (%)'}
)
ax4.set_title('SPY Monthly Return Heatmap (%)', fontsize=13, fontweight='bold', pad=12)
ax4.set_xlabel('Month', fontsize=10)
ax4.set_ylabel('Year',  fontsize=10)
plt.tight_layout()
plt.savefig(OUT_DIR / 'spy_monthly_heatmap.png', dpi=150, bbox_inches='tight')
print('Saved → outputs/spy_monthly_heatmap.png')
plt.show()

Saved → outputs/spy_monthly_heatmap.png


## SQL Block 5 — Worst Calendar Months Across All Tickers

Ranked by total cumulative return within the month — useful for identifying macro stress periods.

In [10]:
# ── 10. DuckDB — worst 15 calendar months ────────────────────────────────
worst_months = con.execute("""
    SELECT
        Ticker,
        STRFTIME(CAST(Date AS DATE), '%Y-%m')       AS month,
        ROUND(SUM(DailyReturn) * 100, 2)            AS total_ret_pct,
        ROUND(MIN(DailyReturn) * 100, 2)            AS worst_day_pct,
        ROUND(MAX("^VIX"), 1)                       AS peak_vix,
        COUNT(*)                                    AS trading_days
    FROM market
    WHERE DailyReturn IS NOT NULL
    GROUP BY Ticker, month
    ORDER BY total_ret_pct ASC
    LIMIT 15
""").df()

print('=== 15 Worst Calendar Months (any ticker) ===')
print(worst_months.to_string(index=False))

=== 15 Worst Calendar Months (any ticker) ===
Ticker   month  total_ret_pct  worst_day_pct  peak_vix  trading_days
   XOM 2020-03         -25.60         -12.22      82.7            22
  AMZN 2022-04         -25.34         -14.05      33.5            20
  AMZN 2018-10         -21.04          -7.82      25.2            23
  AAPL 2018-11         -19.11          -6.63      22.5            21
   JPM 2020-03         -18.22         -14.96      82.7            22
   XOM 2020-02         -16.80          -6.02      40.1            19
   JPM 2022-06         -15.68          -4.60      34.0            21
   XOM 2018-12         -14.93          -3.83      36.1            19
   GLD 2026-03         -14.87          -4.46      31.0            21
   XOM 2020-09         -14.78          -3.21      33.6            21
   XOM 2023-05         -13.48          -3.99      20.1            22
  AMZN 2022-12         -13.42          -3.43      25.0            21
  AAPL 2019-05         -12.75          -5.81      20.5   

In [11]:
print('\n✓ 02_eda complete')
for f in sorted(OUT_DIR.iterdir()):
    print(f'  → {f.name:<40}  {f.stat().st_size/1024:>7.1f} KB')


✓ 02_eda complete
  → anomaly_AAPL.png                            127.3 KB
  → anomaly_AMZN.png                            134.9 KB
  → anomaly_GLD.png                             112.5 KB
  → anomaly_JPM.png                             123.6 KB
  → anomaly_SPY.png                             124.1 KB
  → anomaly_XOM.png                             122.9 KB
  → correlation_heatmap.png                     104.8 KB
  → normalised_price.png                        367.1 KB
  → rolling_volatility.png                      352.7 KB
  → spy_monthly_heatmap.png                     151.9 KB
